In [ ]:
import funciones
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

In [ ]:
img_raw = funciones.cargar_imagen('img/pensamientos.jpg')
# img_raw = funciones.cargar_imagen(r"img/flores de lupino.png")

img_cuantizada, paleta_img, labels, colores_paleta = funciones.cuantizar_imagen(img_raw, n_colores=16)

In [ ]:
labels.shape

In [ ]:
from scipy.ndimage import generic_filter
from collections import Counter
import numpy as np
from scipy import ndimage
import pandas as pd

In [ ]:
# Filtro mayoría
def majority(window):
    return Counter(window).most_common(1)[0][0]

def nearest_color(img):
    # Aplicar vecindad 3x3
    pixeles_filtrados = generic_filter(
        img,
        function=majority,
        size=3,
        mode='nearest'
    )
    return pixeles_filtrados

In [ ]:
def segmentar_regiones(imagen_cuantizada, labels, colores_paleta):
    """
    Identifica regiones conectadas (connected components) para cada color
    
    Args:
        imagen_cuantizada: imagen con colores reducidos
        labels: etiquetas de cluster de K-means (H, W)
        colores_paleta: array con los 20 colores
    
    Returns:
        mapa_regiones: array donde cada región tiene un ID único
        info_regiones: DataFrame con información de cada región
    """
    h, w = labels.shape
    # print(f"Procesando imagen de {h}x{w} píxeles...")
    
    # Crear mapa de regiones (inicialmente vacío)
    mapa_regiones = np.zeros((h, w), dtype=np.int32)
    
    # Lista para almacenar información de cada región
    regiones_info = []
    region_id_global = 1  # Contador global de regiones
    
    # Procesar cada color por separado
    # print(f"\nBuscando regiones conectadas por color...")
    
    for color_idx in range(len(colores_paleta)):
        # Crear máscara binaria para este color
        mascara_color = (labels == color_idx).astype(np.uint8)
        
        # Encontrar componentes conectados (8-connectivity)
        regiones_etiquetadas, num_regiones = ndimage.label(mascara_color)
        
        if num_regiones == 0:
            continue
        
        # print(f"  Color {color_idx + 1:2d} RGB{tuple(colores_paleta[color_idx])}: {num_regiones:3d} regiones")
        
        # Para cada región de este color
        for region_idx in range(1, num_regiones + 1):
            mascara_region = (regiones_etiquetadas == region_idx)
            area = np.sum(mascara_region)
            
            # Asignar ID global único a esta región
            mapa_regiones[mascara_region] = region_id_global
            
            # Guardar información
            regiones_info.append({
                'region_id': region_id_global,
                'color_id': color_idx + 1,  # 1-indexed para el usuario
                'color_rgb': tuple(colores_paleta[color_idx]),
                'area_pixels': area,
                'porcentaje': 100 * area / (h * w)
            })
            
            region_id_global += 1
    
    # Convertir a DataFrame
    df_regiones = pd.DataFrame(regiones_info)
    
    # print(f"\n{'='*60}")
    # print(f"Total de regiones encontradas: {len(df_regiones)}")
    # print(f"{'='*60}")
    
    return mapa_regiones, df_regiones

In [ ]:
Image.fromarray(img_cuantizada).save('output/paso2/filtrada_0.png')

mapa_regiones, df_regiones = segmentar_regiones(img_cuantizada, labels, colores_paleta)

df_filtrar_pixeles = []
n_iteraciones = 50
historial_iteraciones = []
i = 1

df_filtrar_pixeles.append({
    'iteracion': 0,
    'regiones_con_area_1': df_regiones[df_regiones['area_pixels'] == 1].shape[0],
    'total_regiones': df_regiones.shape[0]
})

# for i in range(1, n_iteraciones):
while i <= n_iteraciones:
    print(f"\nIteración {i}")
    #aplicar filtrado
    labels = nearest_color(labels)
    img_filtrada = colores_paleta[labels]

    mapa_regiones, df_regiones = segmentar_regiones(img_filtrada, labels, colores_paleta)    

    n_1px = df_regiones[df_regiones['area_pixels'] == 1].shape[0]
    
    df_filtrar_pixeles.append({
        'iteracion': i,
        'regiones_con_area_1': n_1px,
        'total_regiones': df_regiones.shape[0]
    })

    Image.fromarray(img_filtrada).save(f'output/paso2/filtrada_{i}.png')
    
    #gestion del loop
    historial_iteraciones.append(n_1px)
    if len(historial_iteraciones) > 3:
        historial_iteraciones.pop(0)
    
    if len(historial_iteraciones) == 3 and len(set(historial_iteraciones)) == 1:
        break
    i+=1

# df_regiones[df_regiones['area_pixels'] == 1].shape[0]

df_filtrar_pixeles = pd.DataFrame(df_filtrar_pixeles)
# df_filtrar_pixeles.head()

In [ ]:
df_filtrar_pixeles

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(df_filtrar_pixeles['iteracion'], df_filtrar_pixeles['regiones_con_area_1'], marker='o', label='Regiones con área 1')
plt.plot(df_filtrar_pixeles['iteracion'], df_filtrar_pixeles['total_regiones'], marker='o', label='Total de regiones')

ax = plt.gca()

ax.xaxis.set_major_locator(MultipleLocator(1))

plt.xlabel('Iteración')
plt.ylabel('Número de Regiones')
plt.title('Evolución del Número de Regiones durante el Filtrado')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# df_regiones.describe()
df_regiones[df_regiones['area_pixels'] == 1]


In [ ]:
#TODO: pasar el procesamiento al archivo paso2.py